In [ ]:
import pandas as pd
from openpyxl import load_workbook
import numpy as np  
import os
import sys

# ------------------------------
# Excel Table Extraction
# ------------------------------
def extract_excel_table(filename, sheet_name):
    wb = load_workbook(filename, data_only=True)
    ws = wb[sheet_name] if sheet_name else wb.active
    tables = ws._tables
    if not tables:
        raise ValueError(f"No tables found in sheet '{sheet_name}'")
    tab = list(tables.values())[0]
    ref = tab.ref
    start_cell, end_cell = ref.split(':')
    start_col = ''.join(filter(str.isalpha, start_cell))
    start_row = int(''.join(filter(str.isdigit, start_cell)))
    end_col = ''.join(filter(str.isalpha, end_cell))
    end_row = int(''.join(filter(str.isdigit, end_cell)))
    skiprows = start_row - 1
    nrows = end_row - start_row + 1
    usecols = f"{start_col}:{end_col}"
    df = pd.read_excel(filename, sheet_name=sheet_name, skiprows=skiprows, nrows=nrows, usecols=usecols)
    return df

# read excel sheet into dataframe
# ------------------------------
def read_excel_sheet(filename, sheet_name):
    df = pd.read_excel(filename, sheet_name=sheet_name)
    return df

ROOT = 'D:\\IMADS\\CoffeeGrinder'
SUBDS = 'STWINonGrinder'
metadata_file = os.path.join(ROOT, '2025 dataset planning.xlsx')
base_dir = os.path.join(ROOT, SUBDS)
 
# df = extract_excel_table(metadata_file, 'Config')
# df

df = read_excel_sheet(metadata_file, 'Config')
# keep only until Cut end(s) and remove empty columns
df = df.iloc[:, :10]
if SUBDS == 'STWINonGrinder':
    df = df.drop(columns=['ID acq table'])
    # rename ID acq grinder to ID acq
    df.rename(columns={'ID acq grinder': 'ID acq'}, inplace=True)
elif SUBDS == 'STWINonTable':
    df = df.drop(columns=['ID acq grinder']) 
    # rename ID acq grinder to ID acq
    df.rename(columns={'ID acq table': 'ID acq'}, inplace=True)
df = df.drop(columns=df.columns[-3:])
df

In [ ]:
# ------------------------------
# Dataset Building
# ------------------------------
def build_datasets(df, base_dir):
    dataset_records = []
    background_records = []
 
    df = df[df['ID acq'].notna()]
 
    for _, row in df.iterrows():
        acq_id = row['ID acq']
        file_folder = os.path.join(base_dir, f'{acq_id}', '_Exported')
        mic1_file = os.path.join(file_folder, 'imp23absu_mic.wav')
        mic2_file = os.path.join(file_folder, 'imp34dt05_mic.wav')
 
        is_background = not pd.isna(row['Bkg noise']) and str(row['Bkg noise']).strip() != '\\'
 
        if is_background:
            for mic_file in [mic1_file, mic2_file]:
                background_records.append({
                    'ID acq': acq_id,
                    'file path': mic_file,
                    'pos': row['pos'],
                    'background ID': row['Bkg noise']
                })
        else:
            for mic_file in [mic1_file, mic2_file]:
                dataset_records.append({
                    'ID acq': acq_id,
                    'file path': mic_file,
                    'Model': row['Model'],
                    'Type': row['Type'],
                    'position': row['pos'],
                    'grind size': row['grinder precision (um)']
                })
 
    dataset_df = pd.DataFrame(dataset_records)
    background_df = pd.DataFrame(background_records)
 
    return dataset_df, background_df
 
 
segments_dir = os.path.join(base_dir, 'segments')
 
dataset_df, background_df = build_datasets(df, base_dir)
 
dataset_df.to_csv(os.path.join(base_dir, 'dataset.csv'), index=False)
background_df.to_csv(os.path.join(base_dir, 'background.csv'), index=False)

In [ ]:
# show one example of the dataset_df dataframe as a waveform
import librosa
import numpy as np
import os
import soundfile as sf
import numpy as np
 
def export_segments(df, segments_dir, source_target_flag='source', train_test_flag='train'):
    os.makedirs(segments_dir, exist_ok=True)
 
    # Create a DataFrame to store segments, with same columns as df
    segment_columns = df.columns.tolist()
    segments_df = pd.DataFrame(columns=segment_columns)
   
    cnt = 0
    source_domain_dict = {'grind size': [0,100,200],
                          'position': ['A', 'B', 'C']}
 
    for _, row in df.iterrows():
        y, sr = librosa.load(row['file path'], sr=None)
        if sr != 16000:
            print(f"Warning: {row['file path']} has a sample rate of {sr}. Expected 16000.")
        segment_length = 5 * sr
        segments = [y[i:i + segment_length] for i in range(0, len(y), segment_length)
                    if len(y[i:i + segment_length]) == segment_length]
 
        normal_anomaly_flag = 'normal' if row['Type'] == 'Normal' else 'anomaly'
        mic_type = row['file path'].split('.')[-2].split('\\')[-1]
 
        if row['grind size'] in source_domain_dict['grind size']:
            source_target_flag = 'source'
        else:
            source_target_flag = 'target'
 
        for segment in segments:
            filename = f'section_00_{source_target_flag}_{train_test_flag}_{normal_anomaly_flag}_{str(cnt).zfill(4)}_mic_{mic_type}_pos_{row["position"]}_grind_{row["grind size"]}.wav'
            sf.write(os.path.join(segments_dir, filename), segment, sr)
            entry = {
                'ID acq': row['ID acq'],
                'file path': os.path.join('segments', filename),
                'Model': row['Model'],
                'Type': row['Type'],
                'position': row['position'],
                'grind size': row['grind size'],
                'source_target_flag': source_target_flag,
                'train_test_flag': train_test_flag,
                'normal_anomaly_flag': normal_anomaly_flag,
                'mic_type': mic_type
            }
            segments_df = pd.concat([segments_df, pd.DataFrame([entry])], ignore_index=True)
            cnt += 1
 
    return segments_df
 
# Export segments for dataset
segments_df = export_segments(dataset_df, segments_dir, source_target_flag='source', train_test_flag='train')
# save df to base folder
segments_df.to_csv(os.path.join(base_dir, 'segments.csv'), index=False)
segments_df

In [ ]:
# cycle over segments_df to update anomaly file names with the substitution of 'train' with 'test'

for i, row in segments_df.iterrows():
    if 'anomaly' in row['file path']:
        old_file_name = row['file path']
        new_file_name = old_file_name.replace('train', 'test')
        segments_df.at[i, 'file path'] = new_file_name
        segments_df.at[i, 'train_test_flag'] = 'test'

        os.rename(os.path.join(base_dir, old_file_name), os.path.join(base_dir, new_file_name))
        print(f"Renamed {old_file_name} to {new_file_name}")

# show some updated segments_df dataframe examples
segments_df[segments_df['train_test_flag'] == 'test'].head()

In [ ]:
# convert all column datatype to str
segments_df = segments_df.astype(str)


# concatenate values of the columns 'source_target_flag', 'grind size', 'position', and 'mic_type' into a new column 'target_label'
# using '_' as the separator

segments_df['target_label'] = segments_df['source_target_flag'] + '_' + segments_df['grind size'] + '_' + segments_df['position'] + '_' + segments_df['mic_type']
segments_df

In [ ]:
segments_df['target_label'].unique()

In [ ]:
segments_df_normal = segments_df[segments_df['normal_anomaly_flag'] == 'normal']
segments_df_anomaly = segments_df[segments_df['normal_anomaly_flag'] == 'anomaly']

In [ ]:
# split into train and test sets and stratify based on the 'target_label' column
from sklearn.model_selection import train_test_split

# startify normal source
segments_df_NS = segments_df_normal[segments_df_normal['source_target_flag'] == 'source']
train_df_NS, test_df_NS = train_test_split(segments_df_NS, test_size=50, stratify=segments_df_NS['target_label'], random_state=42)

# train_df_NS['target_label'].value_counts()
test_df_NS['target_label'].value_counts() 
# test_df_NS

In [ ]:
# startify normal target
segments_df_NT = segments_df_normal[segments_df_normal['source_target_flag'] == 'target']
train_df_NT, test_df_NT = train_test_split(segments_df_NT, test_size=50, stratify=segments_df_NT['target_label'], random_state=42)

train_df_NT['target_label'].value_counts() 
test_df_NT['target_label'].value_counts() 
test_df_NT

In [ ]:
segments_df_anomaly

In [ ]:
# Read the CSV file into a pandas DataFrame
df = segments_df_anomaly.copy()

# Define the target number of samples
TARGET_SAMPLES = 100  # Replace with your desired number of samples

# Function to perform stratified sampling
def stratified_sample(df, target_column, target_samples):
    # Calculate the number of samples per class
    class_counts = df[target_column].value_counts()
    class_ratios = class_counts / class_counts.sum()
    samples_per_class = (class_ratios * target_samples).round().astype(int)
    
    # Sample from each class
    sampled_df = pd.concat([
        df[df[target_column] == cls].sample(n=samples_per_class[cls], random_state=42)
        for cls in class_counts.index
    ])
    
    return sampled_df

# Perform stratified sampling
test_df_AST = stratified_sample(df, 'target_label', TARGET_SAMPLES)

# Display the sampled DataFrame
test_df_AST['target_label'].value_counts()
# test_df_AST

In [ ]:
test_df = pd.concat([test_df_NS, test_df_NT, test_df_AST], ignore_index=True)
train_df = pd.concat([train_df_NS, train_df_NT], ignore_index=True)
test_df

In [ ]:
# copy all files in the train_df['file path'] to a new directory called 'train'
train_dir = os.path.join(base_dir,'segments', 'train')
os.makedirs(train_dir, exist_ok=True)
for file in train_df['file path']:
    src = os.path.join(base_dir, file)
    dst = os.path.join(train_dir, os.path.basename(file))
    os.system(f'copy "{src}" "{dst}"')

In [ ]:
# copy all files in the test_df['file path'] to a new directory called 'test'
import shutil  

test_dir = os.path.join(base_dir, 'segments', 'test')
os.makedirs(test_dir, exist_ok=True)

# cycle over test_df and copy the files to the test directory
for i, row in test_df.iterrows():
    old_file_name = row['file path']
    new_file_name = row['file path'].replace('train', 'test')
    test_df.at[i, 'file path'] = new_file_name
    test_df.at[i, 'train_test_flag'] = 'test'

    #update the segments_df dataframe with the new file name
    segments_df.loc[segments_df['file path'] == old_file_name, 'file path'] = new_file_name
    segments_df.loc[segments_df['file path'] == old_file_name, 'train_test_flag'] = 'test'

    # rename the file in the segments directory
    old_file_path = os.path.join(base_dir, old_file_name)
    new_file_path = os.path.join(base_dir, new_file_name)
    os.rename(old_file_path, new_file_path)
    
    shutil.copy(new_file_path, test_dir)

In [ ]:
# count the number of element that contain the 'test' value in segments_df['file path']
test_count = segments_df['file path'].str.contains('test').sum()
print(f"Number of test files: {test_count}")